# Resume Parser

### Importing the necessary libraries

In [ ]:
import re
from datetime import datetime
from time import perf_counter
import pymupdf4llm

import json
from pathlib import Path
from typing import Optional
from pydantic import BaseModel, Field, field_validator


from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

### Data models

In [ ]:
class ContactInfo(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    location: Optional[str] = None
    linkedin: Optional[str] = None
    github: Optional[str] = None


class EducationEntry(BaseModel):
    institution: Optional[str] = None
    degree: Optional[str] = None
    cgpa: Optional[str] = None
    location: Optional[str] = None
    dates: Optional[str] = None
    details: Optional[str] = None


class ExperienceEntry(BaseModel):
    title: Optional[str] = None
    company: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    bullets: list[str] = Field(default_factory=list)


class ProjectEntry(BaseModel):
    name: Optional[str] = None
    description: Optional[str] = None
    tech_stack: list[str] | str = Field(default_factory=list)  # type: ignore

    @field_validator("tech_stack", mode="before")
    @classmethod
    def _split_tech_stack(cls, value):
        """Accept either a list or a comma-separated string."""
        if isinstance(value, str):
            return [s.strip() for s in value.split(",") if s.strip()]
        return value


class SkillsBlock(BaseModel):
    # NOTE: e.g., {"Languages": ["Python", "Java"], "Databases": ["PostgreSQL"]}
    categories: dict[str, list[str]] = Field(default_factory=dict)


class Resume(BaseModel):
    contact: ContactInfo = Field(default_factory=ContactInfo)
    summary: Optional[str] = None
    education: list[EducationEntry] = Field(default_factory=list)
    experience: list[ExperienceEntry] = Field(default_factory=list)
    projects: list[ProjectEntry] = Field(default_factory=list)
    skills: SkillsBlock = Field(default_factory=SkillsBlock)
    certifications: list[str] = Field(default_factory=list)
    awards: list[str] = Field(default_factory=list)

### Setup and config.

In [ ]:
def clean_resume_text(text: str) -> str:
    """Replace emoji/mojibake with explicit labels so the LLM can parse them."""
    # NOTE: Common patterns in the current MIT resume design
    replacements = {
        r"\xc2\xa0": " ",  # NOTE: non-breaking space(s)
        r"�": "",  # NOTE: mojibake char(s)
        r"_|_": "|",  # NOTE: separator(s)
    }

    cleaned = text
    for pattern, repl in replacements.items():
        cleaned = re.sub(pattern, repl, cleaned)

    return cleaned


ip_dirpath = Path("../input")
resume_filepath = ip_dirpath / "Resume_v1.pdf"

print(
    f"I/P filepath: {str(resume_filepath)}"
    if resume_filepath.exists()
    else f"❌ ERROR: I/P file not found!"
)

### Parser setup

In [ ]:
model = "ollama:gemma4:31b-cloud"
model = init_chat_model(model=model)

structured_model = model.with_structured_output(Resume)

In [ ]:
SYS_PROMPT = """You are an expert resume parser. Extract structured data using EXACTLY these field names:
- contact (object with: name, email, phone, location, linkedin, github)
- education items: institution, degree, gpa, start_date, end_date (NOT cgpa, NOT dates)
- experience items: title, company, location, start_date, end_date, bullets
- projects items: name, description, tech_stack, bullets
  - tech_stack MUST be a JSON array of individual strings, e.g., ["Python", "Flask"]
  - NOT a comma-separated string like "Python, Flask"
  - description: a detailed and comprehensive summary of what the project does (REQUIRED, do not leave null)
  - bullets: list EACH individual bullet/paragraph as a SEPARATE string — do NOT merge into description
- skills.categories: DICT mapping category name -> list of skills

Return ONLY valid JSON."""

RESUME_PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        (
            "human",
            "Parse this resume. Extract EVERY bullet point into the `bullets` list — "
            "do NOT merge bullets into a description. Extract contact info from the top of the document. "
            "Use the EXACT field names listed above.\n\n"
            "```markdown\n{resume_text}\n```",
        ),
    ]
)

parser = RESUME_PROMPT | structured_model

### Chunking document

In [ ]:
doc_chunks = pymupdf4llm.to_markdown(
    resume_filepath,
    page_chunks=True,
)

### Parsing document using `ollama`

In [ ]:
t0 = perf_counter()
print("=" * 60)
print("PARSING RESUME")
print("=" * 60)
print(f"Input size: {resume_filepath.stat().st_size:,} bytes")

# NOTE: Concatenate all pages into one markdown blob
print("\n[1/3] Concatenating pages...")
pages = [chunk["text"] for chunk in doc_chunks]  # type: ignore
resume_text = "\n\n---\n\n".join(pages)
resume_text = clean_resume_text(resume_text)
print(f"      Combined text: {len(resume_text):,} chars across {len(pages)} page(s)")

# NOTE: Run the LLM parser
print("\n[2/3] Invoking LLM parser...")
invoke_start = perf_counter()
parsed_resume: Resume = parser.invoke({"resume_text": resume_text})  # type: ignore
invoke_elapsed = perf_counter() - invoke_start
print(f"      Done in {invoke_elapsed:.2f}s")

# NOTE: Summarize results
print("\n[3/3] Parsed summary")
print(f"      Contact   : {parsed_resume.contact.name or '<no name>'}")
print(f"      Education : {len(parsed_resume.education)} entries")
print(f"      Experience: {len(parsed_resume.experience)} entries")
print(f"      Projects  : {len(parsed_resume.projects)} entries")
print(f"      Skills    : {len(parsed_resume.skills.categories)} categories")

# NOTE: Total elapsed
total_elapsed = perf_counter() - t0
print(f"\nTotal elapsed: {total_elapsed:.2f}s")

### Parsing to LLM response to `JSON`

In [ ]:
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
output_path = Path(f"./output/Parsed_Resume_{timestamp}.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(
        dict(parsed_resume),  # type: ignore
        f,
        indent=2,
        ensure_ascii=False,
        default=str,  # NOTE: for any non-serializable values
    )

print(f"Saved to: {output_path.resolve()}")
print(f"Size: {output_path.stat().st_size:,} bytes")